# PipelineWatch-NG — Module 2b: TROPOMI Dry Season Validation
### Re-running SO2 for Oct-Dec 2023 (dry season)

**Purpose:** Validate the 8 DBSCAN fire clusters from Module 2 using TROPOMI SO2.
During the wet season (Jan-Jun), clouds mask most retrievals.
The dry season gives much cleaner SO2 columns.

**Key question:** Do any of the 8 fire clusters show persistent SO2 elevation?
Fire + SO2 co-located = high-confidence illegal refinery confirmation.

---

In [ ]:
import ee, os, json, gc
import numpy as np
import pandas as pd
from datetime import datetime

GEE_PROJECT_ID = "pipelinewatch-ng"
CACHE_DIR  = "../data/cached"
OUTPUT_DIR = "../outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print("GEE connected: " + str(ee.Number(42).getInfo()))
except Exception:
    ee.Authenticate(); ee.Initialize(project=GEE_PROJECT_ID)


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
import gc

OUTPUT_DIR = "../outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def show(filename):
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(); gc.collect()
    display(Image(path))
    print("Saved: " + path)

print("Backend: " + matplotlib.get_backend())


## Cell 3 — Config

In [ ]:
ROI_BOUNDS = [6.50, 5.00, 7.20, 5.80]
ROI = ee.Geometry.Rectangle(ROI_BOUNDS)

# Dry season window
DRY_START = "2023-10-01"
DRY_END   = "2023-12-31"

# Wet season for comparison
WET_START = "2023-01-01"
WET_END   = "2023-06-30"

# Lowered threshold for dry season — smaller ops detectable
SO2_THRESHOLD_DU = 1.5
SO2_MIN_OBS      = 3

print("Dry season : " + DRY_START + " to " + DRY_END)
print("SO2 threshold: " + str(SO2_THRESHOLD_DU) + " DU (lowered from 3.0)")


## Cell 4 — Load fire clusters from Module 2

In [ ]:
cluster_path = os.path.join(CACHE_DIR, "m2_refinery_clusters.csv")
if not os.path.exists(cluster_path):
    print("ERROR: run Module 2 first")
else:
    df_clusters = pd.read_csv(cluster_path)
    print("Loaded " + str(len(df_clusters)) + " fire clusters")
    print(df_clusters[["cluster_id","centroid_lat","centroid_lon","n_hotspots","max_T21_K","risk_label"]].to_string(index=False))

# Build GEE 10km buffer around each cluster centroid for SO2 sampling
cluster_features = []
for _, row in df_clusters.iterrows():
    pt = ee.Geometry.Point([row["centroid_lon"], row["centroid_lat"]])
    cluster_features.append(ee.Feature(pt.buffer(10000), {
        "cluster_id": int(row["cluster_id"]),
        "n_hotspots":  int(row["n_hotspots"]),
        "max_T21_K":   float(row["max_T21_K"]),
        "risk_label":  str(row["risk_label"]),
    }))
fc_clusters = ee.FeatureCollection(cluster_features)
print("Buffer zones built: " + str(len(cluster_features)) + " x 10km radius")


## Cell 5 — TROPOMI SO2 composite (dry season)

In [ ]:
def build_so2_composite(start, end, roi, max_cloud=0.4, label=""):
    col = (ee.ImageCollection("COPERNICUS/S5P/NRTI/L3_SO2")
           .filterDate(start, end).filterBounds(roi)
           .select(["SO2_column_number_density","cloud_fraction"]))
    n = col.size().getInfo()
    print("  " + label + " TROPOMI images: " + str(n))
    def mc(img):
        so2 = img.select("SO2_column_number_density").multiply(2241.5).rename("SO2_DU")
        return so2.updateMask(img.select("cloud_fraction").lt(max_cloud))
    clean = col.map(mc)
    comp = (clean.mean().rename("SO2_mean_DU").clip(roi)
            .addBands(clean.max().rename("SO2_max_DU").clip(roi))
            .addBands(clean.count().rename("SO2_obs_count").clip(roi)))
    return comp, n

print("Building TROPOMI composites...")
gc.collect()
so2_dry, n_dry = build_so2_composite(DRY_START, DRY_END, ROI, max_cloud=0.4, label="Dry")
so2_wet, n_wet = build_so2_composite(WET_START, WET_END, ROI, max_cloud=0.3, label="Wet")

# ROI-level stats
def get_stats(comp, roi):
    return comp.select("SO2_mean_DU").reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.max(),sharedInputs=True)
                          .combine(ee.Reducer.percentile([75,90]),sharedInputs=True),
        geometry=roi, scale=5500, maxPixels=1e9, bestEffort=True).getInfo()

dry_stats = get_stats(so2_dry, ROI)
wet_stats = get_stats(so2_wet, ROI)
print()
print("Dry season SO2: mean=" + str(round(dry_stats.get("SO2_mean_DU_mean",0) or 0,3))
      + "  max=" + str(round(dry_stats.get("SO2_mean_DU_max",0) or 0,3)) + " DU")
print("Wet season SO2: mean=" + str(round(wet_stats.get("SO2_mean_DU_mean",0) or 0,3))
      + "  max=" + str(round(wet_stats.get("SO2_mean_DU_max",0) or 0,3)) + " DU")


## Cell 6 — Sample SO2 at each fire cluster centroid

In [ ]:
print("Sampling SO2 at each cluster centroid (10km buffer)...")
gc.collect()

def sample_so2_at_clusters(so2_comp, fc, scale=5500):
    sampled = so2_comp.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.mean().combine(ee.Reducer.max(),sharedInputs=True),
        scale=scale
    )
    rows = []
    for feat in sampled.getInfo()["features"]:
        p = feat["properties"]
        rows.append({
            "cluster_id":   p.get("cluster_id"),
            "n_hotspots":   p.get("n_hotspots"),
            "max_T21_K":    p.get("max_T21_K"),
            "risk_label":   p.get("risk_label"),
            "SO2_mean_DU":  round(p.get("SO2_mean_DU_mean") or 0, 3),
            "SO2_max_DU":   round(p.get("SO2_mean_DU_max")  or 0, 3),
        })
    return pd.DataFrame(rows)

df_dry = sample_so2_at_clusters(so2_dry, fc_clusters)
df_wet = sample_so2_at_clusters(so2_wet, fc_clusters)

df_dry.columns = ["cluster_id","n_hotspots","max_T21_K","risk_label","SO2_dry_mean","SO2_dry_max"]
df_wet.columns = ["cluster_id","n_hotspots","max_T21_K","risk_label","SO2_wet_mean","SO2_wet_max"]

df_so2 = df_dry.merge(df_wet[["cluster_id","SO2_wet_mean","SO2_wet_max"]], on="cluster_id")
df_so2["SO2_dry_exceeds_threshold"] = df_so2["SO2_dry_mean"] >= SO2_THRESHOLD_DU
df_so2["confirmation"] = df_so2["SO2_dry_exceeds_threshold"].map(
    {True:"CONFIRMED - fire + SO2 co-located", False:"unconfirmed - SO2 below threshold"})

print()
print("=== SO2 Co-location Results ===")
print(df_so2[["cluster_id","n_hotspots","max_T21_K","SO2_dry_mean","SO2_dry_max","confirmation"]].to_string(index=False))
n_confirmed = df_so2["SO2_dry_exceeds_threshold"].sum()
print()
print("Confirmed illegal refinery candidates: " + str(n_confirmed) + " / " + str(len(df_so2)))


## Cell 7 — Dry vs wet season SO2 comparison chart

In [ ]:
import numpy as np
x = np.arange(len(df_so2))
w = 0.35
fig, ax = plt.subplots(figsize=(12,6))
bars1 = ax.bar(x-w/2, df_so2["SO2_wet_mean"], w, label="Wet season (Jan-Jun 2023)",
               color="#378ADD", alpha=0.85)
bars2 = ax.bar(x+w/2, df_so2["SO2_dry_mean"], w, label="Dry season (Oct-Dec 2023)",
               color="#E24B4A", alpha=0.85)
ax.axhline(SO2_THRESHOLD_DU, color="#854F0B", linewidth=2, linestyle="--",
           label="Anomaly threshold (" + str(SO2_THRESHOLD_DU) + " DU)")
ax.set_xticks(x)
ax.set_xticklabels(["C"+str(int(c)) for c in df_so2["cluster_id"]], fontsize=10)
ax.set_xlabel("Fire cluster")
ax.set_ylabel("SO2 column density (Dobson Units)")
ax.set_title("PipelineWatch-NG — SO2 at Fire Clusters: Wet vs Dry Season\nTrans Niger Pipeline corridor  |  Bars above threshold = refinery candidate confirmed")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
for bar in list(bars1)+list(bars2):
    h = bar.get_height()
    if h > 0.05:
        ax.text(bar.get_x()+bar.get_width()/2, h+0.02, str(round(h,2)),
                ha="center", va="bottom", fontsize=7)
show("m2b_so2_cluster_comparison.png")


## Cell 8 — Confirmation scatter map

In [ ]:
fig, ax = plt.subplots(figsize=(10,8))

# Plot all clusters — coloured by confirmation
confirmed   = df_so2[df_so2["SO2_dry_exceeds_threshold"]==True]
unconfirmed = df_so2[df_so2["SO2_dry_exceeds_threshold"]==False]

# Load cluster coordinates
df_clust = pd.read_csv(os.path.join(CACHE_DIR,"m2_refinery_clusters.csv"))
df_so2_mapped = df_so2.merge(df_clust[["cluster_id","centroid_lat","centroid_lon"]],on="cluster_id")

conf_mask   = df_so2_mapped["SO2_dry_exceeds_threshold"]==True
unconf_mask = df_so2_mapped["SO2_dry_exceeds_threshold"]==False

ax.scatter(df_so2_mapped.loc[unconf_mask,"centroid_lon"],
           df_so2_mapped.loc[unconf_mask,"centroid_lat"],
           s=df_so2_mapped.loc[unconf_mask,"n_hotspots"]*30,
           c="#B5D4F4", alpha=0.8, edgecolors="#185FA5", linewidths=1.5,
           label="Unconfirmed (SO2 < " + str(SO2_THRESHOLD_DU) + " DU)", zorder=3)

ax.scatter(df_so2_mapped.loc[conf_mask,"centroid_lon"],
           df_so2_mapped.loc[conf_mask,"centroid_lat"],
           s=df_so2_mapped.loc[conf_mask,"n_hotspots"]*40,
           c="#E24B4A", alpha=0.9, edgecolors="#791F1F", linewidths=1.5,
           marker="*", label="CONFIRMED: fire + SO2 co-located", zorder=5)

for _, row in df_so2_mapped.iterrows():
    ax.annotate("C"+str(int(row["cluster_id"]))+
                "\nSO2="+str(row["SO2_dry_mean"])+"DU",
                xy=(row["centroid_lon"],row["centroid_lat"]),
                xytext=(6,6),textcoords="offset points",fontsize=8,
                bbox=dict(boxstyle="round,pad=0.2",facecolor="white",alpha=0.8))

from matplotlib.patches import Rectangle
ax.add_patch(Rectangle((6.5,5.0),0.7,0.8,linewidth=2,
             edgecolor="#185FA5",facecolor="none",linestyle="--"))
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("PipelineWatch-NG — Fire Cluster SO2 Confirmation Map\nDry season Oct-Dec 2023  |  Star = confirmed illegal refinery candidate")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
show("m2b_confirmation_map.png")


## Cell 9 — Save confirmation results

In [ ]:
out_path = os.path.join(CACHE_DIR, "m2b_so2_confirmation.csv")
df_so2_mapped.to_csv(out_path, index=False)
print("Saved: " + out_path)
print()
print("=== Module 2b Summary ===")
print("Fire clusters analysed : " + str(len(df_so2)))
print("Confirmed (fire+SO2)   : " + str(n_confirmed))
print("Unconfirmed            : " + str(len(df_so2)-n_confirmed))
print()
print("Confirmed clusters:")
if n_confirmed > 0:
    conf = df_so2_mapped[df_so2_mapped["SO2_dry_exceeds_threshold"]==True]
    for _, r in conf.iterrows():
        print("  Cluster " + str(int(r["cluster_id"])) +
              " | lat=" + str(round(r["centroid_lat"],3)) +
              " lon=" + str(round(r["centroid_lon"],3)) +
              " | SO2=" + str(r["SO2_dry_mean"]) + " DU" +
              " | T21=" + str(round(r["max_T21_K"],1)) + "K" +
              " | hotspots=" + str(int(r["n_hotspots"])))
else:
    print("  None above threshold. Try lowering SO2_THRESHOLD_DU to 1.0 or extending to Oct 2023-Mar 2024.")
print()
print("Push data/cached/ and outputs/ to GitHub.")
